# SOMAX: Training & Evaluation

**Step 2 of 2 — Benchmarking, training, and GGUF export**

**Prerequisite:** Run `scripts/download.py` first to download the WAXAL dataset.

**Pipeline:**
1. Setup & Dependencies
2. Baseline Token Fertility
3. Train Router Classifier
4. Train Unified BPE Vocabulary
5. Train LoRA Variant (A–E)
6. Benchmark Fertility Reduction
7. Export to GGUF

## 1. Setup & Dependencies

In [1]:
from huggingface_hub import notebook_login

notebook_login()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


In [2]:
# Check GPU
!nvidia-smi

Mon Apr 13 15:26:45 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   38C    P8             12W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
# Skip torch and psutil (already in Colab)
# Focus on the research-specific extensions
!uv pip install -U transformers datasets accelerate peft bitsandbytes llama-cpp-python python-dotenv

Using Python 3.12.13 environment at: /usr
Resolved 79 packages in 1.63s                                        
⠹ Preparing packages... (40/41)                                                 

In [4]:
import os
from pathlib import Path

if not os.path.exists('somax'):
    !git clone 'https://github.com/attabeezy/somax.git'
    %cd somax
else:
    %cd somax

Cloning into 'somax'...
remote: Enumerating objects: 140, done.
remote: Counting objects: 100% (140/140), done.
remote: Compressing objects: 100% (86/86), done.
remote: Total 140 (delta 73), reused 107 (delta 42), pack-reused 0 (from 0)
Receiving objects: 100% (140/140), 78.70 KiB | 6.05 MiB/s, done.
Resolving deltas: 100% (73/73), done.
/content/somax


In [5]:
import os
from dotenv import load_dotenv

# ── Environment Setup ─────────────────────────────────────
# Load secrets from .env if present (used in VS Code Colab extension)
load_dotenv()

# ── Configuration ─────────────────────────────────────────
LANGUAGE    = "twi"                       # Twi (Akan/Ghana) — sole target language
TRAIN_GROUP = "D"                         # primary hypothesis: TTS → ASR → TTS
BASE_MODEL  = "meta-llama/Llama-3.2-1B"
DATA_DIR    = f"data/{LANGUAGE}"
CONFIG_FILE = "configs/variants.yaml"
# ──────────────────────────────────────────────────────────

HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN:
    print("WARNING: HF_TOKEN not found in environment. Model downloads may fail.")
    print("Please set it in your .env file or environment variables.")

In [11]:
import shutil

def setup_drive() -> str | None:
    """Mount Google Drive and return results base path."""
    try:
        from google.colab import drive
        drive.mount("/content/drive", timeout_ms=120000)
        base = "/content/drive/MyDrive/WAXAL-results"
        os.makedirs(base, exist_ok=True)
        print(f"Google Drive mounted: {base}")
        return base
    except Exception as e:
        print(f"Google Drive not available: {e}")
        return None

def save_to_drive(source_dir: str, drive_base: str | None, label: str) -> None:
    """Copy a local directory to Google Drive."""
    if not drive_base or not os.path.exists(source_dir):
        return
    dest = os.path.join(drive_base, label)
    if os.path.exists(dest):
        shutil.rmtree(dest)
    shutil.copytree(source_dir, dest)
    print(f"Saved to Drive: {dest}")

DRIVE_BASE = setup_drive()

# Download dataset if not present
if not Path(DATA_DIR).exists() or not list(Path(DATA_DIR).glob("*.jsonl")):
    print("Dataset not found. Downloading Twi datasets...")
    !python scripts/download.py --output data/
else:
    print(f"Dataset found: {DATA_DIR}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Google Drive mounted: /content/drive/MyDrive/WAXAL-results
Dataset found: data/twi


## 2. Baseline Token Fertility

In [ ]:
# HuggingFace login already handled in the Setup cell above.

The token has not been saved to the git credentials helper. Pass `add_to_git_credential=True` in this function directly or `--add-to-git-credential` if using via `hf`CLI if you want to set the git credential as well.
Token is valid (permission: fineGrained).
The token `somax` has been saved to /root/.cache/huggingface/stored_tokens
Your token has been saved to /root/.cache/huggingface/token
Login successful.
The current active token is: `somax`


In [7]:
!pip install torchvision --upgrade --index-url https://download.pytorch.org/whl/cu130

Looking in indexes: https://download.pytorch.org/whl/cu130
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.5/7.5 MB 78.2 MB/s eta 0:00:00:00:010:01
  Attempting uninstall: torchvision
    Found existing installation: torchvision 0.25.0+cu128
    Uninstalling torchvision-0.25.0+cu128:
      Successfully uninstalled torchvision-0.25.0+cu128


In [8]:
!python scripts/benchmark_fertility.py \
    --tokenizer {BASE_MODEL} \
    --test-file {DATA_DIR}/pristine_twi_test.jsonl \
    --output results/baseline_fertility.json

Traceback (most recent call last):
  File "/content/somax/scripts/benchmark_fertility.py", line 208, in <module>
    main()
  File "/content/somax/scripts/benchmark_fertility.py", line 161, in main
    raise ValueError(f"No texts loaded from {test_file}")
ValueError: No texts loaded from data/twi/pristine_twi_test.jsonl


## 3. Train Router Classifier

In [ ]:
!python scripts/train_router.py \
    --data {DATA_DIR} \
    --output models/router/ \
    --language {LANGUAGE}

Training router classifier for language: akan
Data: data/akan
Loaded 10107 ASR samples
Loaded 872 TTS samples

Total samples: 10979 (10107 ASR, 872 TTS)
Training classifier...
Cross-val F1 (5-fold): 0.914 ± 0.027

Classifier saved to: models/router/akan_router.pkl

Sanity check:
  'uhm chale me dwo o' -> logic
  'The president delivered a formal address to the assembly' -> robust


## 4. Train Unified BPE Vocabulary

In [ ]:
!python scripts/train_bpe.py \
    --input {DATA_DIR} \
    --output models/tokenizers/ \
    --language {LANGUAGE} \
    --vocab-size 8000

ASR: 10107 samples from data/akan/aka_asr_train.jsonl
TTS: 872 samples from data/akan/twi_tts_train.jsonl
Training unified BPE on 10979 samples (10107 ASR + 872 TTS)...
[00:00:00] Tokenize words                 ██████████████████ 22215    /    22215[00:00:00] Tokenize words                 ██████████████████ 0        /        0
[00:00:00] Count pairs                    ██████████████████ 22215    /    22215
[00:00:00] Compute merges                 ██████████████████ 7889     /     7889
Unified tokenizer saved to: models/tokenizers/akan/unified_tokenizer.json
Tokenizer config saved to: models/tokenizers/akan/tokenizer_config.json
Computing per-stream token statistics...
Stream token stats saved to: models/tokenizers/akan/stream_token_stats.json

Training stats saved to: models/tokenizers/akan/training_stats.json
Vocab size: 8000
Token dominance: {'asr': 5308, 'tts': 1403, 'shared': 1289}
Done.


## 5. Train LoRA Variant

Edit `TRAIN_GROUP` in the config cell above to run a different variant:

| Group | Sequence | Notes |
|-------|----------|-------|
| A | ASR only | |
| B | TTS only | |
| C | Mixed | |
| **D** | TTS → ASR → TTS | Primary hypothesis |
| E | ASR → TTS | |

In [ ]:
!python scripts/train_lora.py \
    --group {TRAIN_GROUP} \
    --model {BASE_MODEL} \
    --data {DATA_DIR} \
    --output checkpoints/ \
    --config {CONFIG_FILE}

Variant:    D
Base model: meta-llama/Llama-3.2-1B
Data:       data/akan
Output:     checkpoints/variant_D
Stages:     3

Loading model: meta-llama/Llama-3.2-1B
`torch_dtype` is deprecated! Use `dtype` instead!
model.safetensors: 100% 2.47G/2.47G [00:27<00:00, 90.1MB/s]
Loading weights: 100% 146/146 [00:02<00:00, 63.93it/s]
generation_config.json: 100% 185/185 [00:00<00:00, 1.06MB/s]
bitsandbytes library load error: libnvJitLink.so.13: cannot open shared object file: No such file or directory
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/bitsandbytes/cextension.py", line 320, in <module>
    lib = get_native_library()
          ^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/bitsandbytes/cextension.py", line 298, in get_native_library
    dll = ct.cdll.LoadLibrary(str(binary_path))
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/ctypes/__init__.py", line 460, in LoadLibrary
    return self._dlltype(nam

In [ ]:
save_to_drive("checkpoints", DRIVE_BASE, "checkpoints")

## 6. Benchmark Fertility Reduction

In [ ]:
!python scripts/benchmark_fertility.py \
    --tokenizer {BASE_MODEL} \
    --waxal-tokenizer models/tokenizers/{LANGUAGE}/unified_tokenizer.json \
    --test-file {DATA_DIR}/pristine_twi_test.jsonl \
    --compare \
    --output results/fertility_comparison.json

## 7. Export to GGUF (for Edge Deployment)

In [ ]:
!python scripts/export_gguf.py \
    --checkpoint checkpoints/variant_{TRAIN_GROUP}/final/ \
    --output models/gguf/ \
    --quantization Q4_K_M

In [ ]:
save_to_drive("results", DRIVE_BASE, "results")
save_to_drive("models", DRIVE_BASE, "models")

## Summary

In [ ]:
import json
from pathlib import Path

print("=" * 50)
print("WAXAL-Dual-Core Pipeline Complete!")
print("=" * 50)
print(f"Language:       {LANGUAGE}")
print(f"Variant:        {TRAIN_GROUP}")

results_path = Path("results/fertility_comparison.json")
if results_path.exists():
    r = json.loads(results_path.read_text())
    print(f"\nBaseline F:     {r['baseline']['fertility_mean']:.3f} tokens/word")
    print(f"WAXAL F:        {r['waxal']['fertility_mean']:.3f} tokens/word")
    print(f"Reduction:      {r['reduction_pct']:.1f}%  (target: ≥30%)")
else:
    baseline_path = Path("results/baseline_fertility.json")
    if baseline_path.exists():
        r = json.loads(baseline_path.read_text())
        print(f"\nBaseline F:     {r['fertility_mean']:.3f} tokens/word")

print("\nNext steps:")
print("  1. Download results from Google Drive")
print("  2. Run scripts/benchmark_inference.py on Dell Latitude 7400")
print("  3. Run scripts/benchmark_fertility.py --compare on edge hardware")